
# GRDP Profiling
Table: `pondolens.dev_bronze.grdp_per_capita`  
Baseline: **296 rows*

Baseline: 114 rows per table

**Goal:** Verify economic measures and compatibility with Q3. This dataset supplies the indicator for poverty incidence adn should help answer Q3.

**Row meaning:** geographic area × year × price basis × geographic level. The amount is economic output per person in Philippine pesos.

## 1. Structure and coverage
Inspect columns, sample records, years, regions and price bases. Philippines is a country-level aggregate, not a region. Keep all source rows and metadata while profiling.


In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.grdp_per_capita;


In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.grdp_per_capita
LIMIT 20;


In [0]:
%sql
SELECT
    year,
    price_basis,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT region) AS geographic_label_count
FROM pondolens.dev_bronze.grdp_per_capita
GROUP BY year, price_basis, geographic_level
ORDER BY TRY_CAST(year AS INT), price_basis, geographic_level;


In [0]:
%sql
SELECT
    region_id,
    region,
    geographic_level,
    COUNT(*) AS row_count,
    MIN(year) AS first_year,
    MAX(year) AS last_year
FROM pondolens.dev_bronze.grdp_per_capita
GROUP BY region_id, region, geographic_level
ORDER BY region;


## 2. Data quality and numeric parsing
Check required fields, blanks/dashes, numeric parsing and rescued data. Counts refer to rows, and 0 means no affected rows. Missing, dash and invalid amounts are counted separately; none should become zero automatically.


In [0]:
%sql
WITH checked AS (
    SELECT *,
        NULLIF(TRIM(grdp_per_capita_amount), '') AS amount_text,
        TRY_CAST(NULLIF(TRIM(grdp_per_capita_amount), '') AS DECIMAL(28, 12)) AS amount_value,
        TRY_CAST(NULLIF(TRIM(year), '') AS INT) AS year_value
    FROM pondolens.dev_bronze.grdp_per_capita
)
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(NULLIF(TRIM(region), '') IS NULL) AS missing_region,
    COUNT_IF(NULLIF(TRIM(region_id), '') IS NULL) AS missing_region_id,
    COUNT_IF(NULLIF(TRIM(price_basis), '') IS NULL) AS missing_price_basis,
    COUNT_IF(NULLIF(TRIM(geographic_level), '') IS NULL) AS missing_geographic_level,
    COUNT_IF(NULLIF(TRIM(unit), '') IS NULL) AS missing_unit,
    COUNT_IF(NULLIF(TRIM(source_file), '') IS NULL) AS missing_source_file,
    COUNT_IF(amount_text IS NULL) AS missing_amount,
    COUNT_IF(amount_text IN ('-', '--', '–', '—')) AS dash_amount,
    COUNT_IF(amount_text IS NOT NULL
        AND amount_text NOT IN ('-', '--', '–', '—')
        AND amount_value IS NULL) AS invalid_amount,
    COUNT_IF(amount_value < 0) AS negative_amount,
    COUNT_IF(amount_value = 0) AS zero_amount,
    COUNT_IF(year_value IS NULL) AS missing_or_invalid_year,
    COUNT_IF(NULLIF(TRIM(_rescued_data), '') IS NOT NULL) AS rescued_rows
FROM checked;


In [0]:
%sql
-- Which rows have an empty region_id?
SELECT
    region,
    geographic_level,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.grdp_per_capita
WHERE NULLIF(TRIM(region_id), '') IS NULL
GROUP BY region, geographic_level
ORDER BY region;


In [0]:
%sql
-- Allowed values: each label should appear in a small, expected set
SELECT 'price_basis' AS column_name, price_basis AS value, COUNT(*) AS row_count FROM pondolens.dev_bronze.grdp_per_capita GROUP BY price_basis
UNION ALL
SELECT 'unit', unit, COUNT(*) FROM pondolens.dev_bronze.grdp_per_capita GROUP BY unit
UNION ALL
SELECT 'geographic_level', geographic_level, COUNT(*) FROM pondolens.dev_bronze.grdp_per_capita GROUP BY geographic_level
UNION ALL
SELECT 'source_table', source_table, COUNT(*) FROM pondolens.dev_bronze.grdp_per_capita GROUP BY source_table
ORDER BY column_name, value;


In [0]:
%sql
WITH checked AS (
    SELECT price_basis,
        TRY_CAST(NULLIF(TRIM(grdp_per_capita_amount), '') AS DECIMAL(28, 12)) AS amount_value
    FROM pondolens.dev_bronze.grdp_per_capita
)
SELECT
    price_basis,
    COUNT(*) AS row_count,
    COUNT(amount_value) AS parsed_rows,
    MIN(amount_value) AS min_amount_php_per_person,
    MAX(amount_value) AS max_amount_php_per_person
FROM checked
GROUP BY price_basis
ORDER BY price_basis;


## 3. Duplicate keys and source metadata
A **record key** identifies one indicator row: region + year + price basis + geographic level. The first check lists repeated record keys. The second checks the normalized regional key proposed for Q3: trimmed, uppercase region + typed year + price basis.

**Empty results are expected for these exception queries:** the check completed but found no repeated keys. This differs from a cell that has no saved output at all. After choosing one price basis, the intended Q3 join is one economic record per verified region/year. These checks do not prove that population or spending joins are safe; their keys and boundaries also need checking.


In [0]:
%sql
SELECT
    region,
    year,
    price_basis,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT grdp_per_capita_amount) AS distinct_amounts,
    COUNT(DISTINCT _run_id) AS distinct_runs
FROM pondolens.dev_bronze.grdp_per_capita
GROUP BY region, year, price_basis, geographic_level
HAVING COUNT(*) > 1
ORDER BY row_count DESC, region, year;


In [0]:
%sql
SELECT
    UPPER(TRIM(region)) AS region_join_label,
    TRY_CAST(year AS INT) AS year_value,
    UPPER(TRIM(price_basis)) AS price_basis_key,
    COUNT(*) AS row_count,
    COUNT(DISTINCT grdp_per_capita_amount) AS distinct_amounts
FROM pondolens.dev_bronze.grdp_per_capita
WHERE LOWER(TRIM(geographic_level)) = 'region'
GROUP BY UPPER(TRIM(region)), TRY_CAST(year AS INT), UPPER(TRIM(price_basis))
HAVING COUNT(*) > 1
ORDER BY row_count DESC, region_join_label, year_value;


In [0]:
%sql
SELECT
    source_name, source_file, source_page, source_table,
    _source_file, _source_sheet, _run_id,
    COUNT(*) AS row_count,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at
FROM pondolens.dev_bronze.grdp_per_capita
GROUP BY source_name, source_file, source_page, source_table,
    _source_file, _source_sheet, _run_id
ORDER BY source_table, first_ingested_at;


## 4. Q3 candidates and geographic compatibility
List 2024–2025 regional records, retaining both price bases for inspection. Choose one basis for the final comparison. Do not join on blank region IDs or combine national and regional rows.

The second query compares labels with the GRDP table only. Zero rows means no unmatched labels under this matching rule; it does not confirm boundaries or population/spending compatibility.


In [0]:
%sql
SELECT
    region_id, region,
    TRY_CAST(year AS INT) AS year,
    price_basis,
    TRY_CAST(NULLIF(TRIM(grdp_per_capita_amount), '') AS DECIMAL(28, 12)) AS grdp_per_capita_php_per_person,
    unit, geographic_level, source_file, source_page, source_table
FROM pondolens.dev_bronze.grdp_per_capita
WHERE TRY_CAST(year AS INT) IN (2024, 2025)
  AND LOWER(TRIM(geographic_level)) = 'region'
  AND UPPER(TRIM(region)) <> 'PHILIPPINES'
ORDER BY region, year, price_basis;


In [0]:
%sql
WITH pc AS (SELECT DISTINCT region AS pc_region FROM pondolens.dev_bronze.grdp_per_capita),
     g AS (SELECT DISTINCT region_name AS grdp_region FROM pondolens.dev_bronze.grdp)
SELECT pc.pc_region, g.grdp_region
FROM pc
FULL OUTER JOIN g
  ON UPPER(TRIM(pc.pc_region)) = UPPER(TRIM(g.grdp_region))
WHERE pc.pc_region IS NULL OR g.grdp_region IS NULL
ORDER BY COALESCE(pc.pc_region, g.grdp_region);


## 5. Findings and Silver treatments

| Check | Finding | Proposed Silver treatment |
|---|---|---|
| Coverage | 296 rows across 2018–2025: 288 regional + 8 national; 18 regions including NIR. | Keep  **2021–2025** data for Q3 and review if 2018-2020 data are still needed. Exclude Philippines from regional comparisons. |
| Price basis | Current prices: 152 rows. Constant 2018 prices: 144 rows. Data includes 2019 and 2020, but no constant-2019 or constant-2020 price series. Philippines has current-price rows only. | Keep both series. Review whether Q3 needs constant 2018 prices. |
| Required values | All 296 amounts and years convert to numbers. No missing checked fields except region ID; no amount blanks/dashes, invalid/zero/negative amounts or rescued rows. | Keep source values. |
| Data types | `year`: STRING. `grdp_per_capita_amount`: STRING. `_ingested_at`: TIMESTAMP. | Cast `year` to INT and amount to DECIMAL(28,12). Keep text fields as STRING and `_ingested_at` as TIMESTAMP. |
| Region IDs | 24 blanks: MIMAROPA 16; Philippines 8. | Review missing IDs and fill using team-agreed codes. Avoid blank IDs to match tables if possible. |
| Duplicate and join keys | No repeats for **region + year + price_basis + geographic_level**, or for regional **trimmed uppercase region + INT year + trimmed uppercase price_basis**. | No duplicate removal needed based on these checks. |
| Amount and unit | **Philippine pesos per person**, not thousands/millions. Current-price range: PHP 53,716.52–631,279.08; constant-2018 range: PHP 51,582.05–522,564.46. | Use parsed numeric amounts. No unit multiplier or further division by population. |
| Source metadata | All 296 rows share one `_run_id` (load identifier). The loaded file is `grdp_per_capita_2018-2025.csv`. `source_page` says Capita; Table 2.1 = current prices, Table 2.2 = constant 2018 prices. `_source_sheet` is empty. | Keep all source and load metadata. Populate `_source_sheet` with 'Capita' as seen on the raw source excel fro the grdp_capita dataset or equate to `source_page` is the same. |
| Geographic compatibility | No unmatched region labels against GRDP. Matching population/spending boundaries has not been confirmed. | Check and review the same years and region boundaries, especially NIR, before joining. |


## 6. Assumptions and open checks

| Interpretation / Question | Evidence | Status |
|---|---|---|
| What codes should replace the blank region IDs? | MIMAROPA has 16 blanks; Philippines has 8. IV-B/PH are proposed codes, not values from this source. | Pending team agreement. |
| Do we need constant 2018 prices for Q3? | Current and constant 2018 prices are available across 2021–2025. No constant-2019/2020 series is present. | Pending analysis choice. |
| Can population and spending be matched to these regions for 2021–2025? | GRDP labels match. Population/spending boundaries and selected join records still need confirmation, especially NIR. | Pending cross-dataset check. |


## 7. Semantic notes

- GRDP per capita is economic output per person, not personal income or poverty incidence.
- Unit: PHP per person. Do not multiply by thousands/millions or divide by population again.
- “Constant 2018 prices” means amounts use 2018 as the price reference; the records still cover multiple years.
- Do not sum or simply average regional per-capita values, or mix Philippines with regional records.
- For Q3, match **2021–2025**, the same region boundaries and one selected price basis.


## 8. Conclusion

- All 296 amounts and years convert to numbers, and both key checks found no repeated records. No amount corrections or duplicate removal are needed.
- Cast year to INT and grdp_per_capita_amount to DECIMAL(28,12). Review the 24 blank region IDs and fill them using team-agreed codes.
- Use 2021–2025 regional data, including Philippines. Review whether 2018–2020 records and constant 2018 prices are needed.
- Confirm that population and spending use compatible region boundaries and years, especially for NIR.
- Retain source and load metadata, including _ingested_at as TIMESTAMP. Populate _source_sheet with Capita after confirming the original worksheet.

